# MotorNet-JAX: Training with Environment API

This notebook demonstrates how to train neural network controllers using the full Environment API,
which includes:
- **Proprioceptive feedback**: Normalized muscle length and velocity
- **Visual feedback**: Fingertip position
- **Configurable delays**: Proprioception and vision delays
- **Observation format**: [goal, vision, proprioception] matching PyTorch MotorNet

In [1]:
import os
import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.')))))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from functools import partial
import optax
import equinox as eqx

from motornet_jax.types import JointState
from motornet_jax.effector import Arm26
from motornet_jax.environment.environment import RandomTargetReach, CenterOutReach
from motornet_jax.policy import GRUPolicy
from motornet_jax.plotor import plot_pos_over_time, plot_training_log

print('All packages imported.')
print(f'JAX version: {jax.__version__}')

# Show device information
print(f'\n--- Device Configuration ---')
print(f'Backend: {jax.default_backend()}')
print(f'Devices: {jax.devices()}')

All packages imported.
JAX version: 0.9.0

--- Device Configuration ---


Metal device set to: Apple M4 Max
Backend: METAL
Devices: [METAL(id=0)]


W0000 00:00:1770002592.035982 26909716 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1770002592.051827 26909716 service.cc:145] XLA service 0x600003e11100 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770002592.051852 26909716 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1770002592.053726 26909716 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1770002592.053735 26909716 mps_client.cc:384] XLA backend will use up to 115448250368 bytes on device 0 for SimpleAllocator.


In [2]:
# === Device Configuration ===
# JAX automatically uses the best available backend. To enable GPU:
#
# For NVIDIA GPUs (CUDA):
#   pip install jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
#
# For Apple Silicon (Metal/MPS):
#   pip install jax-metal
#
# To force a specific device:
#   jax.config.update('jax_default_device', jax.devices('gpu')[0])  # Use first GPU
#   jax.config.update('jax_default_device', jax.devices('cpu')[0])  # Force CPU

# Check if GPU is available and set as default
def setup_device(prefer_gpu=True):
    """Setup JAX device. Returns the device being used."""
    backend = jax.default_backend()
    devices = jax.devices()
    
    if prefer_gpu and backend in ['gpu', 'cuda', 'metal']:
        device = devices[0]
        print(f"Using GPU: {device}")
    else:
        device = jax.devices('cpu')[0]
        if prefer_gpu:
            print(f"GPU not available, using CPU: {device}")
        else:
            print(f"Using CPU: {device}")
    
    return device

device = setup_device(prefer_gpu=True)
print(f"All computations will run on: {device}")

GPU not available, using CPU: TFRT_CPU_0
All computations will run on: TFRT_CPU_0


## I. Setup Environment

Create an Arm26 effector and wrap it in an Environment with:
- Proprioceptive feedback delay (default: 1 timestep)
- Visual feedback delay (default: 1 timestep)

The observation will be:
- goal: (2,) - target position
- vision: (2,) - fingertip position (delayed)
- proprioception: (12,) - normalized muscle length (6) + velocity (6) (delayed)

In [3]:
# Create effector and environment
arm = Arm26(dt=0.01, n_ministeps=1)
arm_params = arm.get_params()

# Environment with delays (in seconds)
env = RandomTargetReach(
    arm,
    max_ep_duration=1.0,
    proprioception_delay=0.01,  # 1 timestep delay
    vision_delay=0.01,          # 1 timestep delay
)
env_params = env.get_params()

print(f"Environment observation dimension: {env.observation_dim}")
print(f"  - Goal: 2")
print(f"  - Vision (fingertip): 2")
print(f"  - Proprioception (muscle len+vel): {2 * arm.n_muscles}")
print(f"  Total: {2 + 2 + 2 * arm.n_muscles}")
print()
print(f"Action dimension: {env.action_dim}")
print(f"Proprioception delay: {env_params.proprioception_delay} timesteps")
print(f"Vision delay: {env_params.vision_delay} timesteps")

JaxRuntimeError: UNKNOWN: -:0:0: error: unknown attribute code: 22
-:0:0: note: in bytecode version 6 produced by: StableHLO_v1.13.0


In [ ]:
# Test environment reset and step
key = jax.random.PRNGKey(0)
env_state, obs, info = env.reset(key, batch_size=4)

print(f"Observation shape: {obs.shape}")
print(f"Goal (target position): {obs[0, :2]}")
print(f"Vision (fingertip): {obs[0, 2:4]}")
print(f"Proprioception (muscle len): {obs[0, 4:10]}")
print(f"Proprioception (muscle vel): {obs[0, 10:16]}")

## II. Policy Network

Create a GRU policy that takes the full observation (goal + vision + proprioception).

In [ ]:
# Policy configuration
obs_dim = env.observation_dim  # 16: goal(2) + vision(2) + proprioception(12)
action_dim = env.action_dim     # 6 muscles
hidden_size = 128

key = jax.random.PRNGKey(42)
policy = GRUPolicy(obs_dim, action_dim, hidden_size=hidden_size, n_gru_layers=1, key=key)

print(f"Policy architecture:")
print(f"  Observation dim: {obs_dim}")
print(f"  Action dim: {action_dim}")
print(f"  Hidden size: {hidden_size}")
n_params = sum(x.size for x in jax.tree_util.tree_leaves(eqx.filter(policy, eqx.is_array)))
print(f"  Number of parameters: {n_params:,}")

## III. Loss Function

Using L1 loss on full trajectory (matching PyTorch MotorNet), plus:
- **Effort cost**: Mean squared muscle activation (metabolic cost proxy)
- **Jerk penalty**: Smoothness regularization (3rd derivative of position)

The policy receives the full observation from the environment at each step.

In [ ]:
def compute_loss(policy, env_state, env_params, arm_params, n_steps=100,
                 effort_weight=1e-1, jerk_weight=1e-4):
    """
    Compute loss with multiple components using Environment API:
    
    1. L1 position error on FULL trajectory
    2. Effort cost: mean squared muscle activation
    3. Jerk penalty: smoothness regularization
    
    The policy receives observations in format:
    [goal, vision (delayed), proprioception (delayed)]
    """
    from motornet_jax.environment.environment import Environment
    
    batch_size = env_state.goal.shape[0]
    hidden = policy.init_hidden(batch_size)
    targets = env_state.goal  # (batch, 2)
    
    def step_fn(carry, _):
        state, hidden = carry
        
        # Get observation from environment (includes delays!)
        obs = Environment.get_obs(state, env_params)
        
        # Get action from policy
        action, new_hidden = policy(obs, hidden)
        
        # Step simulation
        new_effector = Arm26.step(
            state.effector, action,
            jnp.zeros((batch_size, 2)), jnp.zeros((batch_size, 2)),
            arm_params
        )
        
        # Update proprioception and vision
        new_prop = Environment.get_proprioception(new_effector, env_params)
        new_vis = Environment.get_vision(new_effector, env_params)
        
        # Update observation buffer
        new_obs_buffer = Environment.update_obs_buffer(
            state.obs_buffer, new_prop, new_vis, action
        )
        
        # Create new state
        from motornet_jax.environment.environment import EnvState
        new_state = EnvState(
            effector=new_effector,
            goal=state.goal,
            obs_buffer=new_obs_buffer,
            step_count=state.step_count + 1,
            elapsed=state.elapsed + env_params.dt,
        )
        
        return (new_state, new_hidden), (new_effector.fingertip, action)
    
    # Run simulation
    (final_state, _), (trajectory, actions) = jax.lax.scan(
        step_fn, (env_state, hidden), None, length=n_steps
    )
    # trajectory: (n_steps, batch, 2)
    # actions: (n_steps, batch, n_muscles)
    
    # === L1 Position Loss ===
    target_trajectory = jnp.broadcast_to(targets[None, :, :], trajectory.shape)
    l1_per_step = jnp.sum(jnp.abs(trajectory - target_trajectory), axis=-1)
    position_loss = jnp.mean(l1_per_step)
    
    # === Effort Cost ===
    effort = jnp.mean(actions ** 2)
    
    # === Jerk Penalty ===
    velocity = jnp.diff(trajectory, axis=0)
    acceleration = jnp.diff(velocity, axis=0)
    jerk = jnp.diff(acceleration, axis=0)
    jerk_penalty = jnp.mean(jerk ** 2)
    
    # === Total Loss ===
    loss = position_loss + effort_weight * effort + jerk_weight * jerk_penalty
    
    # Metrics
    final_pos = trajectory[-1]
    final_error = jnp.sqrt(jnp.sum((final_pos - targets) ** 2, axis=-1))
    
    return loss, {
        'position_error': jnp.mean(final_error),
        'position_loss': position_loss,
        'effort': effort,
        'jerk': jerk_penalty,
    }

## IV. Training Loop

In [ ]:
# Training configuration
batch_size = 32
n_steps = 100
n_epochs = 1000
learning_rate = 1e-3

# Optimizer with gradient clipping
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate)
)
opt_state = optimizer.init(eqx.filter(policy, eqx.is_array))

@eqx.filter_jit
def train_step(policy, opt_state, env_state):
    """Single training step."""
    def loss_fn(policy):
        return compute_loss(policy, env_state, env_params, arm_params, n_steps)
    
    (loss, metrics), grads = eqx.filter_value_and_grad(loss_fn, has_aux=True)(policy)
    
    updates, opt_state_new = optimizer.update(
        grads, opt_state, eqx.filter(policy, eqx.is_array)
    )
    policy_new = eqx.apply_updates(policy, updates)
    
    return policy_new, opt_state_new, loss, metrics

In [ ]:
# Training loop
key = jax.random.PRNGKey(0)
history = {'loss': [], 'position_error': [], 'effort': [], 'jerk': []}

print("Training with Environment API")
print(f"  Observation: goal + vision + proprioception (dim={env.observation_dim})")
print(f"  Delays: prop={env_params.proprioception_delay}, vis={env_params.vision_delay} timesteps")
print(f"  Loss: L1 position + effort + jerk penalties")
print()

for epoch in range(n_epochs):
    key, reset_key = jax.random.split(key)
    
    # Reset environment (generates random targets)
    env_state, obs, info = env.reset(reset_key, batch_size=batch_size)
    
    # Training step
    policy, opt_state, loss, metrics = train_step(policy, opt_state, env_state)
    
    history['loss'].append(float(loss))
    history['position_error'].append(float(metrics['position_error']))
    history['effort'].append(float(metrics['effort']))
    history['jerk'].append(float(metrics['jerk']))
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d}: loss={loss:.4f}, "
              f"pos_err={metrics['position_error']*100:.2f}cm, "
              f"effort={metrics['effort']:.4f}, "
              f"jerk={metrics['jerk']:.2e}")

print(f"\nFinal position error: {history['position_error'][-1]*100:.2f}cm")
print(f"Final effort: {history['effort'][-1]:.4f}")
print(f"Final jerk: {history['jerk'][-1]:.2e}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].semilogy(history['loss'])
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title('Training Loss (L1 + effort + jerk)')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot([e * 100 for e in history['position_error']])
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Position Error (cm)')
axes[0, 1].set_title('Final Position Error')
axes[0, 1].axhline(y=3, color='g', linestyle='--', alpha=0.5, label='Target: 3cm')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history['effort'])
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Effort (mean squared activation)')
axes[1, 0].set_title('Effort Cost')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].semilogy(history['jerk'])
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Jerk (d³x/dt³)²')
axes[1, 1].set_title('Jerk Penalty (smoothness)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## V. Evaluate Trained Policy

In [ ]:
from motornet_jax.environment.environment import Environment, EnvState

def evaluate_trajectory(policy, env, key, n_steps=100):
    """Generate trajectory using Environment API."""
    env_state, obs, info = env.reset(key, batch_size=1)
    hidden = policy.init_hidden(1)
    target = env_state.goal[0]
    
    positions = [env_state.effector.fingertip[0]]
    actions = []
    
    for _ in range(n_steps):
        obs = Environment.get_obs(env_state, env_params)
        action, hidden = policy(obs, hidden)
        actions.append(action[0])
        
        # Step effector
        new_effector = Arm26.step(
            env_state.effector, action,
            jnp.zeros((1, 2)), jnp.zeros((1, 2)),
            arm_params
        )
        
        # Update observation buffer
        new_prop = Environment.get_proprioception(new_effector, env_params)
        new_vis = Environment.get_vision(new_effector, env_params)
        new_obs_buffer = Environment.update_obs_buffer(
            env_state.obs_buffer, new_prop, new_vis, action
        )
        
        env_state = EnvState(
            effector=new_effector,
            goal=env_state.goal,
            obs_buffer=new_obs_buffer,
            step_count=env_state.step_count + 1,
            elapsed=env_state.elapsed + env_params.dt,
        )
        positions.append(new_effector.fingertip[0])
    
    return jnp.stack(positions), jnp.stack(actions), target

# Evaluate on multiple targets
n_eval = 8
trajectories = []
actions_list = []
targets = []

for i in range(n_eval):
    key = jax.random.PRNGKey(i * 100)
    pos, act, tgt = evaluate_trajectory(policy, env, key)
    trajectories.append(pos)
    actions_list.append(act)
    targets.append(tgt)

targets = jnp.stack(targets)

In [ ]:
# Plot trajectories
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = plt.cm.hsv(np.linspace(0, 1, n_eval))

ax = axes[0]
for i, (traj, target) in enumerate(zip(trajectories, targets)):
    ax.plot(traj[:, 0], traj[:, 1], color=colors[i], linewidth=2, alpha=0.8)
    ax.scatter([target[0]], [target[1]], color=colors[i], s=100, marker='x', zorder=5)

ax.scatter([trajectories[0][0, 0]], [trajectories[0][0, 1]], 
           color='black', s=200, marker='*', zorder=10, label='Start')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Trained Policy: Random Target Reaching')
ax.axis('equal')
ax.grid(True, alpha=0.3)
ax.legend()

# Muscle activations
ax = axes[1]
t = np.arange(n_steps) * arm_params.dt
for m in range(6):
    ax.plot(t, actions_list[0][:, m], label=arm.muscle_names[m], linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Activation')
ax.set_title('Muscle Activations (Trial 0)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compute final errors
errors = []
for traj, target in zip(trajectories, targets):
    final_pos = traj[-1]
    error = float(jnp.sqrt(jnp.sum((final_pos - target) ** 2)))
    errors.append(error)

print("Final position errors:")
for i, error in enumerate(errors):
    print(f"  Trial {i}: {error*100:.2f} cm")
print(f"\n  Mean: {np.mean(errors)*100:.2f} cm")
print(f"  Std:  {np.std(errors)*100:.2f} cm")

## VI. Test with Different Delays

Let's see how performance changes with longer delays.

In [ ]:
# Create environment with longer delays
env_delayed = RandomTargetReach(
    arm,
    max_ep_duration=1.0,
    proprioception_delay=0.05,  # 5 timesteps
    vision_delay=0.05,          # 5 timesteps
)
env_delayed_params = env_delayed.get_params()

print(f"Delayed Environment:")
print(f"  Proprioception delay: {env_delayed_params.proprioception_delay} timesteps ({env_delayed_params.proprioception_delay * arm_params.dt * 1000:.0f}ms)")
print(f"  Vision delay: {env_delayed_params.vision_delay} timesteps ({env_delayed_params.vision_delay * arm_params.dt * 1000:.0f}ms)")

## Summary

This notebook demonstrated training with the full Environment API:

1. **Environment Setup**: Create effector and environment with configurable delays
2. **Observation Format**: goal + vision + proprioception (matching PyTorch MotorNet)
3. **Proprioception**: Normalized muscle length and velocity
4. **Delays**: Configurable proprioception and vision delays
5. **Loss Function**:
   - **L1 position loss**: Full trajectory tracking
   - **Effort cost**: Mean squared muscle activation (metabolic cost proxy)
   - **Jerk penalty**: Smoothness regularization (encourages minimum-jerk trajectories)

### Key Features:

- **Biologically realistic feedback**: Normalized muscle proprioception
- **Sensory delays**: Configurable delays for proprioception and vision
- **PyTorch MotorNet compatibility**: Same observation format and API
- **Biologically plausible costs**: Effort and smoothness penalties encourage realistic movements